# Phase 5: Full Deployment — Gradio App

**Using your existing 3-channel models. Zero training required.**

### What this builds:
- ✅ **Triple CNN Ensemble** (Xception + ResNet50)
- ✅ **Confidence-Weighted Soft Voting**
- ✅ **Image + Video Pipeline**
- ✅ **Grad-CAM Heatmaps**
- ✅ **Gradio Web Interface**

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'gradio', '--quiet'])

CompletedProcess(args=['c:\\Users\\aksha\\AppData\\Local\\Programs\\Python\\Python311\\python.exe', '-m', 'pip', 'install', 'gradio', '--quiet'], returncode=0)

In [6]:
import os, cv2, numpy as np, tensorflow as tf, gradio as gr
import matplotlib.pyplot as plt
from PIL import Image

MODELS_DIR = 'models'
print('TF:', tf.__version__)

print('Loading models...')
model_xcep = tf.keras.models.load_model(os.path.join(MODELS_DIR, '140K_xception_model.keras'), compile=False)
model_res  = tf.keras.models.load_model(os.path.join(MODELS_DIR, '140K_resnet50_model.keras'), compile=False)

MODELS = {
    'Xception': (model_xcep, 256),
    'ResNet50':  (model_res,  224),
}
print('Models loaded.')

MODELS['ResNet50'] = (model_res, 256)


TF: 2.21.0
Loading models...
Models loaded.


In [7]:
def preprocess(img_rgb, size):
    img = cv2.resize(img_rgb, (size, size))
    return img.astype(np.float32) / 255.0

def predict_ensemble(img_rgb):
    probs = {}
    for name, (model, size) in MODELS.items():
        x = np.expand_dims(preprocess(img_rgb, size), 0)
        p = float(model.predict(x, verbose=0)[0][0])
        probs[name] = p
    weights = {n: abs(p - 0.5) * 2 for n, p in probs.items()}
    total_w = sum(weights.values()) + 1e-9
    ensemble = sum(weights[n] * probs[n] for n in probs) / total_w
    probs['Ensemble'] = ensemble
    return probs

def make_gradcam(model, img_rgb, size):
    last_conv = None
    for layer in reversed(model.layers):
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.SeparableConv2D)):
            last_conv = layer.name
            break
    grad_model = tf.keras.Model(inputs=model.input,
                                outputs=[model.get_layer(last_conv).output, model.output])
    x = np.expand_dims(preprocess(img_rgb, size), 0)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(x)
        loss = preds[:, 0]
    grads = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.reduce_sum(tf.multiply(pooled, conv_out), axis=-1)[0]
    cam = np.maximum(cam.numpy(), 0)
    cam = cv2.resize(cam, (size, size))
    cam = (cam - cam.min()) / (cam.max() + 1e-7)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    orig = cv2.resize(img_rgb, (size, size))
    return cv2.addWeighted(orig, 0.6, heatmap, 0.4, 0)

print('Core functions ready.')

Core functions ready.


In [8]:
def analyse_image(pil_img):
    if pil_img is None:
        return 'No image uploaded.', None, None
    img_rgb = np.array(pil_img.convert('RGB'))
    probs = predict_ensemble(img_rgb)
    ens = probs['Ensemble']
    label = '🟢 REAL' if ens > 0.5 else '🔴 FAKE'
    conf  = ens if ens > 0.5 else 1 - ens
    summary = f'{label}  |  Confidence: {conf*100:.1f}%\n'
    for name, p in probs.items():
        bar = '█' * int(p * 20) + '░' * (20 - int(p * 20))
        summary += f'  {name:12s}: [{bar}] {p*100:.1f}%\n'
    cam_img = make_gradcam(model_xcep, img_rgb, 256)
    fig, ax = plt.subplots(figsize=(6, 3))
    names = list(probs.keys())
    vals  = [probs[n] * 100 for n in names]
    colors = ['#2ecc71' if v > 50 else '#e74c3c' for v in vals]
    ax.barh(names, vals, color=colors)
    ax.axvline(50, color='white', linestyle='--', alpha=0.5)
    ax.set_xlim(0, 100)
    ax.set_xlabel('Real Probability (%)')
    ax.set_facecolor('#1a1a2e')
    fig.patch.set_facecolor('#1a1a2e')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    plt.tight_layout()
    return summary, Image.fromarray(cam_img), fig

def analyse_video(video_path, n_frames=15, gamma=0.7):
    if video_path is None:
        return 'No video uploaded.', None, None
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    raw_probs, smoothed, S = [], [], None
    worst_frame, worst_score = None, 1.0
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: continue
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        p = predict_ensemble(img_rgb)['Ensemble']
        raw_probs.append(p)
        S = p if S is None else gamma * p + (1 - gamma) * S
        smoothed.append(S)
        if S < worst_score:
            worst_score, worst_frame = S, img_rgb
    cap.release()
    final_prob = float(np.mean(smoothed)) if smoothed else 0.5
    label = '🟢 REAL' if final_prob > 0.5 else '🔴 FAKE'
    conf  = final_prob if final_prob > 0.5 else 1 - final_prob
    summary = f'{label}  |  Confidence: {conf*100:.1f}%\nFrames analysed: {len(smoothed)}'
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(raw_probs, 'o--', color='#3498db', alpha=0.5, label='Per-Frame')
    ax.plot(smoothed, '-', color='#2ecc71', linewidth=2, label='EMA Smoothed')
    ax.axhline(0.5, color='white', linestyle='--', alpha=0.4, label='Decision Boundary')
    ax.fill_between(range(len(smoothed)), 0, 0.5, alpha=0.1, color='#e74c3c')
    ax.fill_between(range(len(smoothed)), 0.5, 1, alpha=0.1, color='#2ecc71')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Real Probability')
    ax.set_xlabel('Frame Index')
    ax.legend(facecolor='#1a1a2e', labelcolor='white')
    ax.set_facecolor('#1a1a2e')
    fig.patch.set_facecolor('#1a1a2e')
    ax.tick_params(colors='white')
    ax.yaxis.label.set_color('white')
    ax.xaxis.label.set_color('white')
    plt.tight_layout()
    worst_pil = Image.fromarray(worst_frame) if worst_frame is not None else None
    return summary, worst_pil, fig

print('Handlers ready.')

Handlers ready.


In [ ]:
CSS = """
.gradio-container { background: #0f0f1a !important; }
h1, h2, h3, p, label { color: #e0e0ff !important; }
.tab-nav button { color: #a0a0cc !important; }
.tab-nav button.selected { color: #7c6bef !important; border-bottom: 2px solid #7c6bef !important; }
"""

with gr.Blocks(theme=gr.themes.Default(primary_hue='purple', neutral_hue='slate'), css=CSS) as demo:

    gr.Markdown("""
    # 🛡️ Hybrid Deepfake Detection System
    ### Triple CNN Ensemble | Confidence-Weighted Voting | Grad-CAM Explainability
    """)

    with gr.Tabs():

        with gr.Tab('🖼️ Image Detection'):
            with gr.Row():
                with gr.Column(scale=1):
                    img_input = gr.Image(type='pil', label='Upload Face Image', height=300)
                    img_btn   = gr.Button('🔍 Analyse Image', variant='primary')
                with gr.Column(scale=2):
                    img_result = gr.Textbox(label='Result', lines=8)
                    with gr.Row():
                        img_cam  = gr.Image(label='Grad-CAM Heatmap')
                        img_plot = gr.Plot(label='Confidence Breakdown')
            img_btn.click(fn=analyse_image, inputs=[img_input],
                          outputs=[img_result, img_cam, img_plot])

        with gr.Tab('🎬 Video Detection'):
            with gr.Row():
                with gr.Column(scale=1):
                    vid_input   = gr.Video(label='Upload Video')
                    n_frames_sl = gr.Slider(5, 30, value=15, step=1, label='Frames to Sample')
                    gamma_sl    = gr.Slider(0.3, 0.95, value=0.7, step=0.05, label='EMA Gamma')
                    vid_btn     = gr.Button('🔍 Analyse Video', variant='primary')
                with gr.Column(scale=2):
                    vid_result   = gr.Textbox(label='Result', lines=4)
                    vid_worst    = gr.Image(label='Most Suspicious Frame')
                    vid_timeline = gr.Plot(label='Temporal Confidence Timeline')
            vid_btn.click(fn=analyse_video,
                          inputs=[vid_input, n_frames_sl, gamma_sl],
                          outputs=[vid_result, vid_worst, vid_timeline])

    gr.Markdown("""
    ---
    > **Models**: Xception + ResNet50 (140K face dataset)
    > **Voting**: Confidence-Weighted Soft Ensemble |
    > **Explainability**: Grad-CAM
    """)

demo.launch(share=False, server_port=7861)


C:\Users\aksha\AppData\Local\Temp\ipykernel_23912\1516547877.py:8: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Default(primary_hue='purple', neutral_hue='slate'), css=CSS) as demo:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\Users\aksha\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\models\functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer']
Received: inputs=Tensor(shape=(1, 256, 256, 3))
  warnings.warn(msg)
Traceback (most recent call last):
  File "c:\Users\aksha\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\aksha\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\aksha\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\User

: 